# 08 · Score scale cap & VT leverage cap (research IS only)

Other test — **not** a STAR change. Descriptive stats for entry score scales and VT
leverage, then a small sensitivity: score caps at 2.0 / 3.0 and leverage caps at 2.0 / 3.0
via notebook-local monkey-patches (restored in `finally`).

Note: production `VolTargetConfig.max_leverage` defaults to **1.5**. The uncapped arm
raises that ceiling so 2× / 3× caps are meaningful comparisons.

## 0. Imports & Config

In [ ]:
from __future__ import annotations

import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from IPython.display import display

from backtest.s2_coint.diagnosis import enrich_trades, extreme_trades
from backtest.s2_coint.report import load_star_stack, require_star
from backtest.s2_coint.research import (
    DEFAULT_STAR_STACK,
    config_from_stack,
    frozen_pairs_for_universe,
    is_end_for_stack,
    load_s1_weekly,
    load_star_panels,
    load_universe_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    overlay_ols_hedge,
    split_is_oos,
)
from backtest.s2_coint.tearsheet import cvar, fit_mean_abs_score
from strategies.s2_coint.engine import simulate_book
from strategies.s2_coint.metrics import corr_to_s1, metrics_from_returns_inference
from strategies.s2_coint.sizing import pair_scale_from_score

warnings.filterwarnings("ignore", category=FutureWarning)

N_EXTREME = 7
STAR_PATH = DEFAULT_STAR_STACK
stack = load_star_stack(STAR_PATH)
require_star("UNIVERSE_STAR", stack.get("UNIVERSE_STAR"))
require_star("EXIT_STAR", stack.get("EXIT_STAR"))
UNIVERSE = str(stack["UNIVERSE_STAR"])
PAIRS = list(stack.get("PAIRS_STAR") or frozen_pairs_for_universe(UNIVERSE, "1d", root=ROOT))
print("UNIVERSE", UNIVERSE)
print("PAIRS", PAIRS)
print("EXIT_STAR", stack.get("EXIT_STAR"))
print("BREAK_STAR", stack.get("BREAK_STAR"))
print("SIZE_STAR", stack.get("SIZE_STAR"))
print("TREND_STAR", stack.get("TREND_STAR"))
print("VOL_STAR", stack.get("VOL_STAR"))
print("NOTE: STAR stack is read-only in this notebook (other_tests).")

In [ ]:
def _cagr(returns: pd.Series, periods_per_year: float = 252.0) -> float:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    if r.empty:
        return float("nan")
    total = float((1.0 + r).prod())
    years = len(r) / float(periods_per_year)
    if years <= 0 or total <= 0:
        return float("nan")
    return float(total ** (1.0 / years) - 1.0)


def arm_metrics(returns: pd.Series, s1: pd.Series | None = None) -> dict:
    r = pd.to_numeric(returns, errors="coerce").fillna(0.0).astype(float)
    r.index = pd.to_datetime(r.index)
    m = metrics_from_returns_inference(r, periods_per_year=252.0)
    cagr = _cagr(r)
    mdd = float(m.get("max_drawdown", float("nan")))
    calmar = float(cagr / abs(mdd)) if np.isfinite(cagr) and np.isfinite(mdd) and mdd != 0 else float("nan")
    return {
        "ann_sharpe": m.get("ann_sharpe", float("nan")),
        "max_drawdown": mdd,
        "calmar": calmar,
        "cagr": cagr,
        "skew": m.get("skew", float("nan")),
        "excess_kurtosis": m.get("excess_kurtosis", float("nan")),
        "cvar_5pct": cvar(r, alpha=0.05),
        "corr_to_s1": corr_to_s1(r, s1 if s1 is not None else s1_weekly),
        "n_days": m.get("n_days", 0),
    }


def collect_trades(book, panel: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for pid, res in book.pair_results.items():
        if res.trades is None or res.trades.empty:
            continue
        t = res.trades.copy()
        t["pair_id"] = str(pid)
        frames.append(enrich_trades(t, panel, res.returns))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def show_extreme(trades: pd.DataFrame, n: int = N_EXTREME, title: str = "") -> None:
    best, worst = extreme_trades(trades, n=n)
    cols = [
        "pair_id", "side_label", "entry_date", "exit_date", "hold_bars",
        "exit_reason", "pnl_pct", "z_entry", "z_exit", "adf_entry", "adf_exit",
    ]
    if title:
        print(title)
    print(f"=== Top {n} trades ===")
    display(best[cols] if not best.empty else best)
    print(f"=== Bottom {n} trades ===")
    display(worst[cols] if not worst.empty else worst)


def metrics_table(rows: dict[str, dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows).T
    order = [
        "ann_sharpe", "max_drawdown", "calmar", "cagr", "skew",
        "excess_kurtosis", "cvar_5pct", "corr_to_s1", "n_days",
    ]
    cols = [c for c in order if c in df.columns] + [c for c in df.columns if c not in order]
    return df[cols]

import strategies.s2_coint.engine as s2_engine
from dataclasses import replace
from risk.analytics.s1_equities.vol_targeting import VolTargetConfig

In [ ]:
bar = str(stack.get("BAR_STAR") or "1d")
lb = lookbacks_for_bar(bar)
# Prefer cached STAR panels when present; else overlay hedge on the research-IS train panel only.
try:
    train_star, _full_star, _manifest = load_star_panels(
        universe=UNIVERSE, bar=bar, pair_ids=PAIRS, root=ROOT
    )
    is_end = is_end_for_stack(stack, train_star)
    is_raw, _oos_unused = split_is_oos(train_star, is_end=is_end)
    del _oos_unused
    panel_src = "cached star train"
except (FileNotFoundError, ValueError) as exc:
    print("star panels unavailable (", type(exc).__name__, ") — using universe train panel")
    train, _full = load_universe_panels(UNIVERSE, bar, PAIRS, root=ROOT)
    is_end = is_end_for_stack(stack, train)
    is_raw, _oos_unused = split_is_oos(train, is_end=is_end)
    del _oos_unused
    panel_src = "universe train"

hedge = str(stack.get("HEDGE_STAR") or "ols")
# Skip re-overlay when star cache already has z / adf columns.
need_overlay = "z" not in is_raw.columns or "adf_pvalue" not in is_raw.columns
if need_overlay:
    if hedge == "kalman":
        is_panel = overlay_kalman_hedge(
            is_raw,
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
    else:
        is_panel = overlay_ols_hedge(
            is_raw,
            ols_window=lb["ols_window"],
            z_window=lb["z_window"],
            hl_window=lb["hl_window"],
            adf_window=lb["adf_window"],
        )
else:
    is_panel = is_raw.copy()

is_panel = is_panel.loc[is_panel["pair_id"].astype(str).isin(PAIRS)].copy()
is_panel["date"] = pd.to_datetime(is_panel["date"])

mean_abs = fit_mean_abs_score(is_panel, score_column="z")
s1_weekly = load_s1_weekly(ROOT)
cfg_star = config_from_stack(stack)
print("panel_src", panel_src, "need_overlay", need_overlay)
print("bar", bar, "is_end", is_end)
print("IS rows", len(is_panel), "pairs", is_panel["pair_id"].nunique())
print("frozen IS mean(|z|)", round(mean_abs, 4))
print("cfg", cfg_star)

## 1. Score Scale Descriptive Stats

In [ ]:
book0 = simulate_book(is_panel, cfg_star, mean_abs_score=mean_abs)
trades0 = collect_trades(book0, is_panel)

scale_rows = []
for row in trades0.itertuples(index=False):
    pid = str(row.pair_id)
    sig = pd.Timestamp(row.signal_date) if pd.notna(row.signal_date) else pd.NaT
    g = is_panel.loc[is_panel["pair_id"].astype(str) == pid].sort_values("date")
    if pd.isna(sig) or g.empty:
        continue
    hit = g.loc[pd.to_datetime(g["date"]) == sig]
    if hit.empty:
        hit = g.loc[pd.to_datetime(g["date"]) <= sig].tail(1)
    if hit.empty:
        continue
    z = float(hit["z"].iloc[0])
    adf = float(hit["adf_pvalue"].iloc[0]) if "adf_pvalue" in hit.columns else float("nan")
    scale = pair_scale_from_score(z, adf, size_mode=str(cfg_star.size_mode), mean_abs_score=mean_abs)
    scale_rows.append({"pair_id": pid, "entry_date": row.entry_date, "z": z, "scale": scale, "pnl_pct": row.pnl_pct})
scale_df = pd.DataFrame(scale_rows)
print("score scale percentiles")
display(scale_df["scale"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("scale"))
display(scale_df.groupby("pair_id")["scale"].describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(scale_df["scale"].dropna(), bins=30, color="#1f4e79", alpha=0.85)
axes[0].axvline(2, color="C1", ls="--"); axes[0].axvline(3, color="C3", ls="--")
axes[0].set_title("Entry scale histogram")
axes[0].grid(True, alpha=0.3)
for pid, g in scale_df.groupby("pair_id"):
    axes[1].plot(pd.to_datetime(g["entry_date"]), g["scale"], ".", label=pid, alpha=0.7)
axes[1].axhline(2, color="C1", ls="--"); axes[1].axhline(3, color="C3", ls="--")
axes[1].legend(); axes[1].set_title("Scale at entry over time"); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Score Scale Cap Test

In [ ]:
results = {}
trades_by = {}

def run_with_score_cap(name: str, cap: float | None) -> None:
    orig = s2_engine.pair_scale_from_score

    def capped(score, adf_pvalue, *, size_mode, mean_abs_score=1.0):
        s = orig(score, adf_pvalue, size_mode=size_mode, mean_abs_score=mean_abs_score)
        return float(s) if cap is None else float(min(s, float(cap)))

    s2_engine.pair_scale_from_score = capped
    try:
        book = simulate_book(is_panel, cfg_star, mean_abs_score=mean_abs)
    finally:
        s2_engine.pair_scale_from_score = orig
    results[name] = arm_metrics(book.returns)
    trades_by[name] = collect_trades(book, is_panel)
    print(name, results[name])

run_with_score_cap("score_uncapped", None)
run_with_score_cap("score_cap_2", 2.0)
run_with_score_cap("score_cap_3", 3.0)
display(metrics_table({k: results[k] for k in ("score_uncapped", "score_cap_2", "score_cap_3")}))
for arm in ("score_uncapped", "score_cap_2", "score_cap_3"):
    show_extreme(trades_by[arm], title=arm)

## 3. VT Leverage Descriptive Stats

In [ ]:
lev = book0.leverage
if lev is None or lev.empty:
    print("No leverage series — check VOL_STAR=s1_vt")
else:
    lev = lev.astype(float).dropna()
    print("NOTE: engine VolTargetConfig defaults max_leverage=1.5 under STAR.")
    display(lev.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("leverage"))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
    lev.plot(ax=axes[0], title="STAR VT leverage", color="#1f4e79"); axes[0].grid(True, alpha=0.3)
    axes[1].hist(lev, bins=30, color="#1f4e79", alpha=0.85); axes[1].set_title("Histogram"); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 4. VT Leverage Cap Test

In [ ]:
def run_with_lev_cap(name: str, cap: float | None) -> None:
    """Uncapped raises VolTargetConfig.max_leverage; capped arms then clip."""
    orig = s2_engine.leverage_from_history

    def wrapped(past_returns, cfg, *, prev_leverage=None):
        cfg2 = replace(cfg, max_leverage=100.0 if cap is None else max(float(cap), float(cfg.min_leverage)))
        lev = orig(past_returns, cfg2, prev_leverage=prev_leverage)
        if cap is not None:
            lev = min(float(lev), float(cap))
        return float(lev)

    s2_engine.leverage_from_history = wrapped
    try:
        book = simulate_book(is_panel, cfg_star, mean_abs_score=mean_abs)
    finally:
        s2_engine.leverage_from_history = orig
    results[name] = arm_metrics(book.returns)
    trades_by[name] = collect_trades(book, is_panel)
    lev = book.leverage
    print(name, results[name])
    if lev is not None and len(lev):
        print("  lev max", float(lev.max()), "p99", float(lev.quantile(0.99)))


# Reference: STAR production path (max_leverage=1.5 default)
results["vt_star_default"] = arm_metrics(book0.returns)
trades_by["vt_star_default"] = trades0

run_with_lev_cap("vt_uncapped", None)
run_with_lev_cap("lev_cap_2", 2.0)
run_with_lev_cap("lev_cap_3", 3.0)
display(metrics_table({
    k: results[k]
    for k in ("vt_star_default", "vt_uncapped", "lev_cap_2", "lev_cap_3")
}))
for arm in ("vt_star_default", "vt_uncapped", "lev_cap_2", "lev_cap_3"):
    show_extreme(trades_by[arm], title=arm)


## 5. Comparison

In [ ]:
focus = [
    "score_uncapped", "score_cap_2", "score_cap_3",
    "vt_star_default", "vt_uncapped", "lev_cap_2", "lev_cap_3",
]
tbl = metrics_table({k: results[k] for k in focus if k in results})
display(tbl)

fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
for ax, col, title in zip(
    axes.ravel(),
    ["ann_sharpe", "excess_kurtosis", "cvar_5pct", "calmar"],
    ["Sharpe", "Excess kurtosis", "CVaR 5%", "Calmar"],
):
    tbl[col].plot(kind="bar", ax=ax, color="#1f4e79", title=title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for arm in focus:
    if arm in trades_by:
        show_extreme(trades_by[arm], title=f"Comparison extremes: {arm}")


## 6. Summary

Does a score cap at 2 or 3 reduce kurtosis with acceptable Sharpe cost?
Does VT uncapped vs cap-2 vs cap-3 matter given STAR already defaults to max_leverage=1.5?
Candidate for a later sizing / risk hypothesis?